# Olist E-Commerce Analytics

EDA → product metrics → RFM → cohort retention.

Полный пайплайн также доступен как скрипт: `scripts/run_analysis.py`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'scripts'))

from run_analysis import (
    load_raw, build_fact, data_quality_report, compute_kpis,
    monthly_dynamics, category_analysis, region_analysis,
    delivery_analysis, rfm_analysis, cohort_retention,
    outlier_price_analysis, write_business_insights, export_powerbi_tables,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Load & data quality

In [ ]:
tables = load_raw()
for name, df in tables.items():
    print(f'{name:12s} {df.shape}')

fact = build_fact(tables)
dq = data_quality_report(tables, fact)
dq

In [ ]:
tables['orders']['order_status'].value_counts(normalize=True).mul(100).round(2)

## 2. Core KPIs

In [ ]:
kpis, delivered, orders_df = compute_kpis(fact)
kpis

## 3. Sales dynamics, categories, regions

In [ ]:
monthly = monthly_dynamics(delivered)
monthly.tail()

In [ ]:
cat = category_analysis(delivered)
cat.head(10)

In [ ]:
region = region_analysis(delivered)
region.head(10)

## 4. Delivery quality & outliers

In [ ]:
late_by_state, delivery_outlier_pct, rev_cmp = delivery_analysis(
    orders_df, delivered, tables['reviews']
)
print('Delivery IQR outlier %:', round(delivery_outlier_pct, 2))
display(rev_cmp)
late_by_state.head(10)

In [ ]:
price_stats = outlier_price_analysis(delivered)
price_stats

## 5. RFM segmentation

In [ ]:
rfm, seg = rfm_analysis(delivered)
seg

## 6. Cohort retention

In [ ]:
pivot, cohort = cohort_retention(delivered)
pivot.head(10)

## 7. Export + findings

In [ ]:
export_powerbi_tables(delivered, rfm, orders_df)
print(write_business_insights(
    kpis, cat, region, seg, pivot, late_by_state, rev_cmp, price_stats, delivery_outlier_pct
))